# Gatekeeper, giai đoạn 0 (§37), phần A: Kaggle có chấm được task SWE-smith với cùng điểm không?

**Trước khi chạy** (khung Settings bên phải): **Internet: On**, **Accelerator: None**.

Chạy ô 1, 2, rồi ô 3 (**khởi động**, trả về ngay). Sau đó bấm chạy ô 3b (**xem tiến độ**) mỗi 1–2 phút cho tới khi thấy `FINISHED`. Làm tương tự với ô 4 và 4b.
Việc nặng chạy nền, tách khỏi notebook; **không ô nào phải chạy lâu**. Nếu kernel bị khởi động lại: chạy lại ô 1 rồi ô xem tiến độ.
Không ô nào in khóa hay thông tin tài khoản. Kết quả ở `/kaggle/working/`: `k0a.json`, `k0b_kaggle.jsonl`.

In [ ]:
# Ô 1 — cài đặt vào đúng Python của notebook (khoảng 2–3 phút; chạy lại an toàn)
%env UDOCKER_DIR=/kaggle/working/udocker
%pip -q install "swesmith==0.0.9" "swebench==4.1.0" udocker unidiff
import subprocess  # noqa: E402
import sys  # noqa: E402

UD = "import sys; from udocker.maincmd import main; sys.argv[0] = 'udocker'; sys.exit(main())"
r = subprocess.run(
    [sys.executable, "-c", UD, "--allow-root", "install"], capture_output=True, text=True
)
print("udocker install:", r.returncode, (r.stdout + r.stderr).strip()[-200:])
!docker info > /dev/null 2>&1 && echo "Docker daemon: CO" || echo "Docker daemon: KHONG"
!df -h /kaggle/working | tail -1
print("python:", sys.executable)

In [ ]:
# ruff: noqa: F811 (every cell redefines the same helpers on purpose: a cell must run alone)
import os
import subprocess
import sys

W = "/kaggle/working"
os.environ["UDOCKER_DIR"] = f"{W}/udocker"
CODE = next(
    (
        f"{W}/{d}"
        for d in ("gatekeeper/kaggle/scripts", "gatekeeper/scripts")
        if os.path.exists(f"{W}/{d}/udocker_shim.py")
    ),
    None,
)

# The job's own heartbeat (written by the job, not by this kernel): time, RAM, disk every 20 s.
BEAT = (
    "while kill -0 $JOB 2>/dev/null; do "
    'echo "  [beat $(date +%T)] $(free -g | awk \'/Mem/{print "RAM free " $7 " GB"}\') '
    '$(df -h /kaggle/working | awk \'NR==2{print "working used " $3}\')"; sleep 20; done'
)


def start(name, cmd, cwd=W):
    """Start a step in the background, fully detached, unless it is running or finished."""
    log, pidf, donef = f"{W}/{name}.log", f"{W}/{name}.pid", f"{W}/{name}.done"
    if os.path.exists(donef):
        return print(f"{name}: already finished; run the status cell")
    if os.path.exists(pidf):
        try:
            os.kill(int(open(pidf).read()), 0)
            return print(f"{name}: still running; run the status cell")
        except (OSError, ValueError):
            pass
    job = " ".join(cmd)
    script = f'({job}) & JOB=$!; {BEAT} & wait $JOB; echo "exit code: $?"; touch {donef}'
    p = subprocess.Popen(
        ["/bin/bash", "-c", script],
        cwd=cwd,
        stdout=open(log, "a"),
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    open(pidf, "w").write(str(p.pid))
    print(f"started {name} (pid {p.pid}); now run the status cell below, again every minute or two")


def status(name, lines=40):
    log, pidf, donef = f"{W}/{name}.log", f"{W}/{name}.pid", f"{W}/{name}.done"
    if not os.path.exists(log):
        return print(f"{name}: not started")
    alive = False
    if os.path.exists(pidf):
        try:
            os.kill(int(open(pidf).read()), 0)
            alive = True
        except (OSError, ValueError):
            pass
    state = "FINISHED" if os.path.exists(donef) else ("RUNNING" if alive else "STOPPED (died?)")
    text = open(log).read().splitlines()
    print(f"== {name}: {state}; log has {len(text)} lines; last {lines}:")
    print("\n".join(text[-lines:]))


# Ô 2 — mã và dữ liệu (repo công khai; không cần đăng nhập)
REPO = "dinhthithuthuyduphong-bot/gatekeeper"
BRANCH = "claude/gatekeeper-13-research-nxytoh"  # Kaggle work lives here until it runs
os.chdir(W)
if not os.path.isdir("gatekeeper"):
    url = f"https://github.com/{REPO}"
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, url], check=True)
else:
    subprocess.run(["git", "-C", "gatekeeper", "pull", "-q"], check=False)
RAW = f"https://raw.githubusercontent.com/{REPO}"
files = {
    "swesmith_pallets__jinja.jsonl": "data-snapshots/data/instances/swesmith_pallets__jinja.jsonl",
    "f1_probe.json": "trial/results/f1_probe.json",
    "f1_skill_probe.json": "trial/results/f1_skill_probe.json",
}
os.makedirs("k0data", exist_ok=True)
for name, path in files.items():
    dest = f"k0data/{name}"
    if not os.path.exists(dest):
        subprocess.run(["curl", "-sSfL", "-o", dest, f"{RAW}/{path}"], check=True)
    print(name, os.path.getsize(dest), "byte")
head = subprocess.run(
    ["git", "-C", "gatekeeper", "log", "--oneline", "-1"], capture_output=True, text=True
)
print(head.stdout)
CODE = f"{W}/gatekeeper/kaggle/scripts"
print("code:", CODE, os.path.exists(f"{CODE}/k0a_probe.py"))

In [ ]:
import os
import subprocess
import sys

W = "/kaggle/working"
os.environ["UDOCKER_DIR"] = f"{W}/udocker"
CODE = next(
    (
        f"{W}/{d}"
        for d in ("gatekeeper/kaggle/scripts", "gatekeeper/scripts")
        if os.path.exists(f"{W}/{d}/udocker_shim.py")
    ),
    None,
)

# The job's own heartbeat (written by the job, not by this kernel): time, RAM, disk every 20 s.
BEAT = (
    "while kill -0 $JOB 2>/dev/null; do "
    'echo "  [beat $(date +%T)] $(free -g | awk \'/Mem/{print "RAM free " $7 " GB"}\') '
    '$(df -h /kaggle/working | awk \'NR==2{print "working used " $3}\')"; sleep 20; done'
)


def start(name, cmd, cwd=W):
    """Start a step in the background, fully detached, unless it is running or finished."""
    log, pidf, donef = f"{W}/{name}.log", f"{W}/{name}.pid", f"{W}/{name}.done"
    if os.path.exists(donef):
        return print(f"{name}: already finished; run the status cell")
    if os.path.exists(pidf):
        try:
            os.kill(int(open(pidf).read()), 0)
            return print(f"{name}: still running; run the status cell")
        except (OSError, ValueError):
            pass
    job = " ".join(cmd)
    script = f'({job}) & JOB=$!; {BEAT} & wait $JOB; echo "exit code: $?"; touch {donef}'
    p = subprocess.Popen(
        ["/bin/bash", "-c", script],
        cwd=cwd,
        stdout=open(log, "a"),
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    open(pidf, "w").write(str(p.pid))
    print(f"started {name} (pid {p.pid}); now run the status cell below, again every minute or two")


def status(name, lines=40):
    log, pidf, donef = f"{W}/{name}.log", f"{W}/{name}.pid", f"{W}/{name}.done"
    if not os.path.exists(log):
        return print(f"{name}: not started")
    alive = False
    if os.path.exists(pidf):
        try:
            os.kill(int(open(pidf).read()), 0)
            alive = True
        except (OSError, ValueError):
            pass
    state = "FINISHED" if os.path.exists(donef) else ("RUNNING" if alive else "STOPPED (died?)")
    text = open(log).read().splitlines()
    print(f"== {name}: {state}; log has {len(text)} lines; last {lines}:")
    print("\n".join(text[-lines:]))


# Ô 3 — K0a: khởi động (trả về ngay). Việc chạy nền 5–15 phút.
IMAGE = "swebench/swesmith.x86_64.pallets_1776_jinja.ada0a9a6"
start("k0a", [sys.executable, f"{CODE}/k0a_probe.py", "--image", IMAGE, "--out-dir", W])

In [ ]:
import os
import subprocess
import sys

W = "/kaggle/working"
os.environ["UDOCKER_DIR"] = f"{W}/udocker"
CODE = next(
    (
        f"{W}/{d}"
        for d in ("gatekeeper/kaggle/scripts", "gatekeeper/scripts")
        if os.path.exists(f"{W}/{d}/udocker_shim.py")
    ),
    None,
)

# The job's own heartbeat (written by the job, not by this kernel): time, RAM, disk every 20 s.
BEAT = (
    "while kill -0 $JOB 2>/dev/null; do "
    'echo "  [beat $(date +%T)] $(free -g | awk \'/Mem/{print "RAM free " $7 " GB"}\') '
    '$(df -h /kaggle/working | awk \'NR==2{print "working used " $3}\')"; sleep 20; done'
)


def start(name, cmd, cwd=W):
    """Start a step in the background, fully detached, unless it is running or finished."""
    log, pidf, donef = f"{W}/{name}.log", f"{W}/{name}.pid", f"{W}/{name}.done"
    if os.path.exists(donef):
        return print(f"{name}: already finished; run the status cell")
    if os.path.exists(pidf):
        try:
            os.kill(int(open(pidf).read()), 0)
            return print(f"{name}: still running; run the status cell")
        except (OSError, ValueError):
            pass
    job = " ".join(cmd)
    script = f'({job}) & JOB=$!; {BEAT} & wait $JOB; echo "exit code: $?"; touch {donef}'
    p = subprocess.Popen(
        ["/bin/bash", "-c", script],
        cwd=cwd,
        stdout=open(log, "a"),
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    open(pidf, "w").write(str(p.pid))
    print(f"started {name} (pid {p.pid}); now run the status cell below, again every minute or two")


def status(name, lines=40):
    log, pidf, donef = f"{W}/{name}.log", f"{W}/{name}.pid", f"{W}/{name}.done"
    if not os.path.exists(log):
        return print(f"{name}: not started")
    alive = False
    if os.path.exists(pidf):
        try:
            os.kill(int(open(pidf).read()), 0)
            alive = True
        except (OSError, ValueError):
            pass
    state = "FINISHED" if os.path.exists(donef) else ("RUNNING" if alive else "STOPPED (died?)")
    text = open(log).read().splitlines()
    print(f"== {name}: {state}; log has {len(text)} lines; last {lines}:")
    print("\n".join(text[-lines:]))


# Ô 3b — xem tiến độ K0a. Chạy lại ô này mỗi 1–2 phút cho tới khi thấy FINISHED.
status("k0a")
if os.path.exists(f"{W}/k0a_mode.txt"):
    print("==> mode:", open(f"{W}/k0a_mode.txt").read().strip() or "(none found)")

In [ ]:
import os
import subprocess
import sys

W = "/kaggle/working"
os.environ["UDOCKER_DIR"] = f"{W}/udocker"
CODE = next(
    (
        f"{W}/{d}"
        for d in ("gatekeeper/kaggle/scripts", "gatekeeper/scripts")
        if os.path.exists(f"{W}/{d}/udocker_shim.py")
    ),
    None,
)

# The job's own heartbeat (written by the job, not by this kernel): time, RAM, disk every 20 s.
BEAT = (
    "while kill -0 $JOB 2>/dev/null; do "
    'echo "  [beat $(date +%T)] $(free -g | awk \'/Mem/{print "RAM free " $7 " GB"}\') '
    '$(df -h /kaggle/working | awk \'NR==2{print "working used " $3}\')"; sleep 20; done'
)


def start(name, cmd, cwd=W):
    """Start a step in the background, fully detached, unless it is running or finished."""
    log, pidf, donef = f"{W}/{name}.log", f"{W}/{name}.pid", f"{W}/{name}.done"
    if os.path.exists(donef):
        return print(f"{name}: already finished; run the status cell")
    if os.path.exists(pidf):
        try:
            os.kill(int(open(pidf).read()), 0)
            return print(f"{name}: still running; run the status cell")
        except (OSError, ValueError):
            pass
    job = " ".join(cmd)
    script = f'({job}) & JOB=$!; {BEAT} & wait $JOB; echo "exit code: $?"; touch {donef}'
    p = subprocess.Popen(
        ["/bin/bash", "-c", script],
        cwd=cwd,
        stdout=open(log, "a"),
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    open(pidf, "w").write(str(p.pid))
    print(f"started {name} (pid {p.pid}); now run the status cell below, again every minute or two")


def status(name, lines=40):
    log, pidf, donef = f"{W}/{name}.log", f"{W}/{name}.pid", f"{W}/{name}.done"
    if not os.path.exists(log):
        return print(f"{name}: not started")
    alive = False
    if os.path.exists(pidf):
        try:
            os.kill(int(open(pidf).read()), 0)
            alive = True
        except (OSError, ValueError):
            pass
    state = "FINISHED" if os.path.exists(donef) else ("RUNNING" if alive else "STOPPED (died?)")
    text = open(log).read().splitlines()
    print(f"== {name}: {state}; log has {len(text)} lines; last {lines}:")
    print("\n".join(text[-lines:]))


# Ô 4 — K0b: khởi động (trả về ngay). Việc chạy nền 30–60 phút.
mode = open(f"{W}/k0a_mode.txt").read().strip() if os.path.exists(f"{W}/k0a_mode.txt") else ""
if not mode:
    raise SystemExit("K0a chưa xong hoặc chưa tìm được cách chạy: chạy ô 3b và gửi k0a.log")
os.environ["UDOCKER_EXECMODE"] = mode
os.makedirs(f"{W}/k0run", exist_ok=True)
D = f"{W}/k0data"
args = [sys.executable, f"{CODE}/k0_stage0.py", "--rows", f"{D}/swesmith_pallets__jinja.jsonl"]
args += ["--results", f"{D}/f1_probe.json", f"{D}/f1_skill_probe.json"]
args += ["--controls", "20", "--out", f"{W}/k0b_kaggle.jsonl"]
start("k0b", args, cwd=f"{W}/k0run")  # SWE-smith writes its logs under ./logs/run_evaluation

In [ ]:
import os
import subprocess
import sys

W = "/kaggle/working"
os.environ["UDOCKER_DIR"] = f"{W}/udocker"
CODE = next(
    (
        f"{W}/{d}"
        for d in ("gatekeeper/kaggle/scripts", "gatekeeper/scripts")
        if os.path.exists(f"{W}/{d}/udocker_shim.py")
    ),
    None,
)

# The job's own heartbeat (written by the job, not by this kernel): time, RAM, disk every 20 s.
BEAT = (
    "while kill -0 $JOB 2>/dev/null; do "
    'echo "  [beat $(date +%T)] $(free -g | awk \'/Mem/{print "RAM free " $7 " GB"}\') '
    '$(df -h /kaggle/working | awk \'NR==2{print "working used " $3}\')"; sleep 20; done'
)


def start(name, cmd, cwd=W):
    """Start a step in the background, fully detached, unless it is running or finished."""
    log, pidf, donef = f"{W}/{name}.log", f"{W}/{name}.pid", f"{W}/{name}.done"
    if os.path.exists(donef):
        return print(f"{name}: already finished; run the status cell")
    if os.path.exists(pidf):
        try:
            os.kill(int(open(pidf).read()), 0)
            return print(f"{name}: still running; run the status cell")
        except (OSError, ValueError):
            pass
    job = " ".join(cmd)
    script = f'({job}) & JOB=$!; {BEAT} & wait $JOB; echo "exit code: $?"; touch {donef}'
    p = subprocess.Popen(
        ["/bin/bash", "-c", script],
        cwd=cwd,
        stdout=open(log, "a"),
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    open(pidf, "w").write(str(p.pid))
    print(f"started {name} (pid {p.pid}); now run the status cell below, again every minute or two")


def status(name, lines=40):
    log, pidf, donef = f"{W}/{name}.log", f"{W}/{name}.pid", f"{W}/{name}.done"
    if not os.path.exists(log):
        return print(f"{name}: not started")
    alive = False
    if os.path.exists(pidf):
        try:
            os.kill(int(open(pidf).read()), 0)
            alive = True
        except (OSError, ValueError):
            pass
    state = "FINISHED" if os.path.exists(donef) else ("RUNNING" if alive else "STOPPED (died?)")
    text = open(log).read().splitlines()
    print(f"== {name}: {state}; log has {len(text)} lines; last {lines}:")
    print("\n".join(text[-lines:]))


# Ô 4b — xem tiến độ K0b. Chạy lại ô này mỗi vài phút cho tới khi thấy FINISHED.
status("k0b", lines=30)

## Gửi lại

`/kaggle/working/k0a.log`, `k0a.json`, `k0b.log`, `k0b_kaggle.jsonl` (khung Output, nút Download), hoặc dán phần in ra của ô 3b và 4b khi đã `FINISHED`.

Phần B (K0c, K0d: gọi model bằng hạn mức AI) chạy ở notebook tạo từ https://www.kaggle.com/benchmarks/tasks/new, bằng ô `kaggle/notebooks/k0cd_benchmarks_cell.py`.